In [2]:
import pandas as pd
import matplotlib.pyplot as plt

Matplotlib is building the font cache; this may take a moment.


In [13]:
dataset = pd.read_csv('https://drive.google.com/uc?id=1c50z3TtNR4DRHCjz4vsPvV6pYnnsem4V')
dataset.head()

,text_id,text,label
0,1,INFO RESMI: Selamat No.Anda terpilih pemenang ...,scam
1,2,"Yth. Nasabah, kartu ATM Anda terblokir karena ...",scam
2,3,BUTUH DANA CPT? Ajukan pinjaman di KREDIT KILA...,scam
3,4,"Mba, tlg isikan pulsa 50rb ke nomor ini dulu y...",scam
4,5,"Sdr/i, kami menemukan paket mencurigakan atas ...",scam


In [4]:
fraud_data = dataset[dataset['label'] == 'scam']

# Preprocess Data

In [5]:
!pip install sentence-transformers scikit-learn
!pip install indonlp  
!pip install Sastrawi  

     |████████████████████████████████| 488 kB 518 kB/s eta 0:00:01
     |████████████████████████████████| 11.1 MB 1.3 MB/s eta 0:00:01
     |████████████████████████████████| 12.0 MB 2.1 MB/s eta 0:00:01
     |████████████████████████████████| 73.6 MB 1.4 MB/s eta 0:00:01    |████▏                           | 9.6 MB 1.6 MB/s eta 0:00:40     |████▍                           | 10.2 MB 1.1 MB/s eta 0:00:59     |████▉                           | 11.2 MB 1.1 MB/s eta 0:00:58
     |████████████████████████████████| 30.3 MB 3.1 MB/s eta 0:00:01     |███████████▊                    | 11.1 MB 361 kB/s eta 0:00:54
     |████████████████████████████████| 515 kB 2.6 MB/s eta 0:00:01
     |████████████████████████████████| 78 kB 7.8 MB/s eta 0:00:011
     |████████████████████████████████| 308 kB 2.9 MB/s eta 0:00:01
     |████████████████████████████████| 200 kB 1.8 MB/s eta 0:00:01
     |████████████████████████████████| 174 kB 556 kB/s eta 0:00:01
     |████████████████████████████████| 73 kB 

In [7]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re
import string
from indoNLP.preprocessing import replace_slang
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

# Initialize Sastrawi
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

stopword_factory = StopWordRemoverFactory()
stopword_remover = stopword_factory.create_stop_word_remover()

In [8]:
def preprocess_indonesian_text(text, remove_punct=False, remove_stops=False, apply_stemming=False):

    if pd.isna(text) or text == '':
        return ''
    
    # Convert ke lowercase
    text = text.lower()
    
    # Normalize slang menggunakan IndoNLP
    text = replace_slang(text)
    
    # Normalize whitespace (remove extra spaces, tabs, newlines)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove punctuation
    if remove_punct:
        text = text.translate(str.maketrans('', '', string.punctuation))
        text = re.sub(r'\s+', ' ', text).strip()  
        
    # Remove stopwords 
    if remove_stops:
        text = stopword_remover.remove(text)
    
    # Apply stemming
    if apply_stemming:
        text = stemmer.stem(text)
    
    return text


# Test preprocessing 
test_texts = [
    "Mba, tlg isikan pulsa 50rb ke nomor ini dulu ya",
    "Selamat! Anda menang hadiah 10jt. Klik link ini",
    "Bos, ini nomor baruku. Tolong pinjemin dulu 2jt yah"
]

print("Preprocessing Examples (IndoNLP + Sastrawi):")
print("=" * 80)
for test_text in test_texts:
    preprocessed = preprocess_indonesian_text(test_text)
    print(f"Original:     {test_text}")
    print(f"Preprocessed: {preprocessed}")
    print("-" * 80)

Preprocessing Examples (IndoNLP + Sastrawi):
Original:     Mba, tlg isikan pulsa 50rb ke nomor ini dulu ya
Preprocessed: mbak, tolong isikan pulsa 50rb ke nomor ini dulu ya
--------------------------------------------------------------------------------
Original:     Selamat! Anda menang hadiah 10jt. Klik link ini
Preprocessed: selamat! anda menang hadiah 10jt. klik link ini
--------------------------------------------------------------------------------
Original:     Bos, ini nomor baruku. Tolong pinjemin dulu 2jt yah
Preprocessed: bos, ini nomor baruku. tolong pinjemin dulu 2jt ya
--------------------------------------------------------------------------------


# Load Model

In [9]:
from transformers import AutoTokenizer, AutoModel
import torch

# Load IndoBERT model and tokenizer
model_name = 'indobenchmark/indobert-base-p1'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

/Users/qika/Documents/Code/college/coolyeah/NLP/ScamDetection_NLP/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/qika/Documents/Code/college/coolyeah/NLP/ScamDetection_NLP/venv/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Get Embedding

In [10]:
def get_embeddings(texts):
    if isinstance(texts, str):
        texts = [texts]
    
    # Tokenize teks
    encoded_input = tokenizer(texts, padding=True, truncation=True, 
                             max_length=512, return_tensors='pt')
    
    # Dapatkan output model
    with torch.no_grad():
        model_output = model(**encoded_input)
    
    # Mean pooling 
    attention_mask = encoded_input['attention_mask']
    token_embeddings = model_output.last_hidden_state  # Shape: (batch_size, seq_len, hidden_size)
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    

    embeddings = torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)
    
    return embeddings.numpy()

# Compute Semantic Similarity

In [11]:
# Preprocess scam data texts
fraud_texts = fraud_data['text'].fillna('').tolist()
fraud_texts_processed = [preprocess_indonesian_text(text) for text in fraud_texts]

# Encode scam data texts
fraud_embeddings = get_embeddings(fraud_texts_processed)

In [12]:
def compute_semantic_similarity(input_text, top_k=10):
    
    # Preprocess input text 
    input_text_processed = preprocess_indonesian_text(input_text)

    # Encode input text menggunakan IndoBERT
    input_embedding = get_embeddings([input_text_processed])
    
    # Compute cosine similarity
    similarities = cosine_similarity(input_embedding, fraud_embeddings)[0]
    
    # get top-k indices
    top_indices = similarities.argsort()[::-1][:top_k]
    
    results = []
    for idx in top_indices:
        results.append({
            "text": fraud_data.iloc[idx],
            "similarity": float(similarities[idx])
        })
    
    # Return top k results
    return results